# 07. FAISS Vector Search for Fast Recommendation

## Objective

In the previous notebook, we generated dense user embeddings using Matrix Factorization (Truncated SVD).

These embeddings represent each user in a low-dimensional latent space where users with similar movie preferences are located close to one another.

Searching through all users one by one becomes inefficient as the dataset grows. Instead, we use Facebook AI Similarity Search (FAISS), a highly optimized library for fast nearest-neighbor search.

In this notebook we will:

- Load user embeddings generated from SVD
- Build a FAISS index
- Store all user vectors inside the index
- Search for the most similar users efficiently
- Compare similarity scores
- Prepare the foundation for large-scale recommendation systems

---

## Why FAISS?

Without FAISS:

- Compare one user with every other user.
- Time Complexity ≈ O(N)

With FAISS:

- Uses optimized vector indexing.
- Extremely fast even for millions of vectors.
- Used in modern recommendation systems like:
    - Netflix
    - Spotify
    - Amazon
    - YouTube
    - Facebook
    - Instagram

---

## Output of this Notebook

By the end of this notebook we will have:

- Dense user embeddings
- FAISS Index
- Nearest Neighbor Search
- Top-K Similar Users
- Ready-to-use Vector Search Engine

In [1]:
# Uncomment if FAISS is not installed

# !pip install faiss-cpu

In [2]:
!pip install faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 10.8 MB/s  0:00:020.1 MB/s eta 0:00:012


In [3]:
import pandas as pd
import numpy as np

import faiss

import pickle

print("Libraries Loaded Successfully")

Libraries Loaded Successfully


In [4]:
# Load User Embeddings

user_embeddings = pd.read_csv(
    "models/user_embeddings.csv"
)

print(user_embeddings.shape)

user_embeddings.head()

FileNotFoundError: [Errno 2] No such file or directory: 'models/user_embeddings.csv'

In [5]:
import pandas as pd

user_embeddings = pd.read_csv("models/user_embeddings.csv")

print(user_embeddings.shape)

user_embeddings.head()

(943, 50)


,0,1,2,3,4,5,6,7,8,9,...,40,41,42,43,44,45,46,47,48,49
0,42.043422,-1.664974,-1.504603,-13.232066,-1.373398,-2.455140,10.683289,2.048045,-6.288519,1.783210,...,-0.927891,-3.030461,-7.503306,-1.177249,2.890222,-0.253742,-2.720006,-6.574519,-4.582502,-9.237924
1,8.949614,11.332479,11.399634,2.693323,2.249586,-0.805017,2.221449,0.668281,-3.767594,-0.441966,...,1.054280,-1.255287,2.426386,-1.536785,1.911899,0.088747,-0.518464,2.282758,-0.895738,-0.966301
2,3.618921,6.239959,5.156245,3.988138,-7.171478,-0.538339,0.598842,0.324907,2.405869,-0.676893,...,-0.034776,-0.855813,-1.090589,0.809254,0.327687,-0.056035,1.243082,-0.437090,1.368236,-1.345938
3,3.847190,5.089566,2.793742,2.845738,-4.194924,1.743970,1.402905,0.965034,2.239359,0.770970,...,0.855991,0.726215,-0.170992,1.185312,0.378650,0.019840,-0.530024,2.591905,1.274696,0.838490
4,20.988305,-2.175145,-10.056875,-2.370013,-1.517414,1.975597,12.086783,-9.001514,1.365370,2.134960,...,-1.525999,-1.906463,-2.662789,-0.383496,-2.270307,1.898602,-1.900444,-1.435556,-1.241544,-0.620153


# Building the FAISS Index

In this section, we build a FAISS (Facebook AI Similarity Search) index using the user embeddings generated from Matrix Factorization (Truncated SVD).

FAISS is a high-performance library developed by Meta AI for efficient similarity search over dense vectors. Instead of comparing a query vector with every embedding manually, FAISS organizes vectors into an optimized index, allowing nearest-neighbor searches to be performed in milliseconds.

### Objectives

- Load the generated user embeddings.
- Convert embeddings into the format required by FAISS.
- Build an IndexFlatL2 index.
- Store all user embeddings inside the FAISS index.
- Verify that the index contains all users.

This index will later be used to retrieve the most similar users based on embedding distance.

In [7]:
import faiss
import numpy as np
import pandas as pd

print("Libraries Loaded Successfully")

Libraries Loaded Successfully


In [8]:
user_embeddings = pd.read_csv("models/user_embeddings.csv")

print(user_embeddings.shape)

user_embeddings.head()

(943, 50)


,0,1,2,3,4,5,6,7,8,9,...,40,41,42,43,44,45,46,47,48,49
0,42.043422,-1.664974,-1.504603,-13.232066,-1.373398,-2.455140,10.683289,2.048045,-6.288519,1.783210,...,-0.927891,-3.030461,-7.503306,-1.177249,2.890222,-0.253742,-2.720006,-6.574519,-4.582502,-9.237924
1,8.949614,11.332479,11.399634,2.693323,2.249586,-0.805017,2.221449,0.668281,-3.767594,-0.441966,...,1.054280,-1.255287,2.426386,-1.536785,1.911899,0.088747,-0.518464,2.282758,-0.895738,-0.966301
2,3.618921,6.239959,5.156245,3.988138,-7.171478,-0.538339,0.598842,0.324907,2.405869,-0.676893,...,-0.034776,-0.855813,-1.090589,0.809254,0.327687,-0.056035,1.243082,-0.437090,1.368236,-1.345938
3,3.847190,5.089566,2.793742,2.845738,-4.194924,1.743970,1.402905,0.965034,2.239359,0.770970,...,0.855991,0.726215,-0.170992,1.185312,0.378650,0.019840,-0.530024,2.591905,1.274696,0.838490
4,20.988305,-2.175145,-10.056875,-2.370013,-1.517414,1.975597,12.086783,-9.001514,1.365370,2.134960,...,-1.525999,-1.906463,-2.662789,-0.383496,-2.270307,1.898602,-1.900444,-1.435556,-1.241544,-0.620153


## Why Convert to NumPy?

FAISS works directly with NumPy arrays rather than Pandas DataFrames.

Although our embeddings are currently stored in a DataFrame, we need to convert them into a NumPy array of type `float32`, since FAISS only accepts vectors in this format.

The conversion does not change the values—it simply prepares the data for efficient indexing and similarity search.

In [9]:
embedding_vectors = user_embeddings.values.astype("float32")

print(embedding_vectors.shape)
print(type(embedding_vectors))

(943, 50)
<class 'numpy.ndarray'>


# 07. FAISS Vector Search

## Objective

In this notebook, we build a high-speed vector search engine using Facebook AI Similarity Search (FAISS).

Instead of comparing one user with every other user manually, FAISS indexes user embeddings and performs extremely fast nearest-neighbor search.

### What we will do

- Load User Embeddings
- Load User Mapping
- Build a FAISS Index
- Store Embeddings inside the Index
- Query Similar Users
- Retrieve Top-K Nearest Users
- Interpret the Results

### Why FAISS?

Traditional similarity search computes distances against every vector, which becomes slow for large datasets.

FAISS organizes vectors into an efficient index so that searching takes milliseconds even for millions of vectors.

This technique is widely used in:

- Recommendation Systems
- Semantic Search
- Retrieval-Augmented Generation (RAG)
- ChatGPT Memory
- Vector Databases
- Image Search

In [10]:
import pandas as pd
import numpy as np
import faiss

print("Libraries Loaded Successfully")

Libraries Loaded Successfully


In [11]:
user_embeddings = pd.read_csv("models/user_embeddings.csv")

print(user_embeddings.shape)

user_embeddings.head()

(943, 50)


,0,1,2,3,4,5,6,7,8,9,...,40,41,42,43,44,45,46,47,48,49
0,42.043422,-1.664974,-1.504603,-13.232066,-1.373398,-2.455140,10.683289,2.048045,-6.288519,1.783210,...,-0.927891,-3.030461,-7.503306,-1.177249,2.890222,-0.253742,-2.720006,-6.574519,-4.582502,-9.237924
1,8.949614,11.332479,11.399634,2.693323,2.249586,-0.805017,2.221449,0.668281,-3.767594,-0.441966,...,1.054280,-1.255287,2.426386,-1.536785,1.911899,0.088747,-0.518464,2.282758,-0.895738,-0.966301
2,3.618921,6.239959,5.156245,3.988138,-7.171478,-0.538339,0.598842,0.324907,2.405869,-0.676893,...,-0.034776,-0.855813,-1.090589,0.809254,0.327687,-0.056035,1.243082,-0.437090,1.368236,-1.345938
3,3.847190,5.089566,2.793742,2.845738,-4.194924,1.743970,1.402905,0.965034,2.239359,0.770970,...,0.855991,0.726215,-0.170992,1.185312,0.378650,0.019840,-0.530024,2.591905,1.274696,0.838490
4,20.988305,-2.175145,-10.056875,-2.370013,-1.517414,1.975597,12.086783,-9.001514,1.365370,2.134960,...,-1.525999,-1.906463,-2.662789,-0.383496,-2.270307,1.898602,-1.900444,-1.435556,-1.241544,-0.620153


## Loading User Mapping

The embedding matrix uses encoded user indices (0 to 942).

The mapping file links each embedding index back to the original MovieLens user ID.

This allows us to interpret search results using the original user identifiers.

In [12]:
user_mapping = pd.read_csv("models/user_mapping.csv")

print(user_mapping.shape)

user_mapping.head()

(943, 1)


,user_id
0,1
1,2
2,3
3,4
4,5


# Building the FAISS Index

FAISS (Facebook AI Similarity Search) requires the embedding vectors to be stored in a specialized index.

Before building the index, we convert the embedding matrix into a NumPy array with `float32` datatype because FAISS only accepts floating-point vectors.

The index will allow us to perform efficient nearest-neighbor searches among all users based on their latent feature representations.

In [13]:
# Convert DataFrame to NumPy array

embedding_matrix = user_embeddings.values.astype("float32")

print(embedding_matrix.shape)
print(type(embedding_matrix))

(943, 50)
<class 'numpy.ndarray'>


# Creating a FAISS Index

FAISS provides several types of indices.

For this project, we use **IndexFlatL2**, which performs an exact nearest-neighbor search using Euclidean (L2) distance.

Although there are faster approximate indices for very large datasets, `IndexFlatL2` is ideal for learning and works well for the MovieLens dataset.

The dimensionality of the index must match the dimensionality of the embedding vectors (50 dimensions in our case).

In [14]:
# Number of embedding dimensions

dimension = embedding_matrix.shape[1]

# Create FAISS Index

index = faiss.IndexFlatL2(dimension)

print(index)

<faiss.swigfaiss.IndexFlatL2; proxy of <Swig Object of type 'faiss::IndexFlatL2 *' at 0x71adec05c6f0> >


# Adding Embeddings to the Index

After creating the FAISS index, we insert all user embedding vectors into it.

Each row of the embedding matrix represents one user.

Once added, FAISS builds an internal structure that enables very fast similarity searches.

The total number of vectors stored in the index can be verified using the `ntotal` attribute.

In [15]:
# Add all embeddings

index.add(embedding_matrix)

print("Total vectors stored:", index.ntotal)

Total vectors stored: 943


# Querying Similar Users

Now that the FAISS index has been built, we can perform similarity searches.

The search process works as follows:

1. Select a query user.
2. Extract that user's embedding vector.
3. Ask FAISS to find the K nearest neighbors.
4. FAISS returns:
   - Distances between the query user and the neighbors.
   - Indices of the most similar users.

Since the query user is already present in the index, the first returned result will always be the query user itself with a distance of **0**.

In [16]:
# =====================================
# Search Similar Users
# =====================================

query_user = 0      # Encoded User Index
k = 6               # Retrieve 6 users (including itself)

distances, indices = index.search(
    embedding_matrix[query_user].reshape(1, -1),
    k
)

print("Distances:")
print(distances)

print("\nNearest User Indices:")
print(indices)

Distances:
[[   0.     1013.2916 1053.2314 1076.0922 1095.6117 1100.3049]]

Nearest User Indices:
[[  0 513 267 867 197  63]]


# Understanding the Search Results

FAISS returns two arrays:

## Distances

The first array contains the Euclidean (L2) distance between the query user and each retrieved user.

- Smaller distance → More similar users.
- Larger distance → Less similar users.

The first distance is always **0** because the query user is compared with itself.

## Indices

The second array contains the encoded user indices corresponding to the nearest neighbors.

These indices can be mapped back to the original MovieLens user IDs using the user mapping file.

In [17]:
# =====================================
# Display Similar Users
# =====================================

print("Query User (Encoded):", query_user)
print()

for rank, idx in enumerate(indices[0]):
    
    original_user = user_mapping.iloc[idx]["user_id"]

    print(f"Rank {rank + 1}")
    print(f"Encoded User  : {idx}")
    print(f"Original User : {original_user}")
    print(f"Distance      : {distances[0][rank]:.4f}")
    print("-" * 40)

Query User (Encoded): 0

Rank 1
Encoded User  : 0
Original User : 1
Distance      : 0.0000
----------------------------------------
Rank 2
Encoded User  : 513
Original User : 514
Distance      : 1013.2916
----------------------------------------
Rank 3
Encoded User  : 267
Original User : 268
Distance      : 1053.2314
----------------------------------------
Rank 4
Encoded User  : 867
Original User : 868
Distance      : 1076.0922
----------------------------------------
Rank 5
Encoded User  : 197
Original User : 198
Distance      : 1095.6117
----------------------------------------
Rank 6
Encoded User  : 63
Original User : 64
Distance      : 1100.3049
----------------------------------------


# Why are the FAISS Distances So Large?

You may notice that the nearest-neighbor distances are around **1000** instead of values between **0 and 1**.

This is completely normal.

## Why?

FAISS is using **Euclidean (L2) Distance** on the raw SVD embeddings.

Each user embedding contains **50 numerical features**, and the Euclidean distance is computed across all of them.

Since the embedding values are not normalized, the squared differences accumulate across all 50 dimensions, resulting in relatively large distance values.

The important point is **not the absolute distance**, but the **relative ordering**.

- Smaller distance → More similar users
- Larger distance → Less similar users

Therefore, the nearest users returned by FAISS are still the correct nearest neighbors.

In later stages, we can normalize the embeddings to use cosine similarity if required, but for this project, the default L2 distance is perfectly acceptable.

# Creating a Reusable FAISS Search Function

Instead of writing the search code repeatedly, we create a reusable function.

The function accepts:

- Encoded User ID
- Number of neighbors (Top-K)

It returns:

- Similar User IDs
- Original MovieLens User IDs
- Euclidean Distances

This makes our recommendation system modular and reusable in future notebooks.

In [18]:
# =====================================
# FAISS Search Function
# =====================================

def search_similar_users(user_index, k=5):

    distances, indices = index.search(
        embedding_matrix[user_index].reshape(1, -1),
        k
    )

    results = pd.DataFrame({
        "Encoded_User": indices[0],
        "Original_User": user_mapping.iloc[indices[0]]["user_id"].values,
        "Distance": distances[0]
    })

    return results

# Testing the Search Function

We now test the reusable search function.

This confirms that the FAISS index is working correctly and that the helper function can retrieve the nearest users in a clean tabular format.

In [19]:
search_similar_users(0, k=10)

,Encoded_User,Original_User,Distance
0,0,1,0.000000
1,513,514,1013.291626
2,267,268,1053.231445
3,867,868,1076.092163
4,197,198,1095.611694
5,63,64,1100.304932
6,737,738,1123.578369
7,805,806,1134.929565
8,520,521,1144.164795
9,915,916,1175.496338


# Saving the FAISS Index

Building an index can be computationally expensive for large datasets.

Instead of rebuilding it every time, FAISS allows us to save the index to disk.

The saved index can later be loaded instantly without recomputing embeddings or rebuilding the search structure.

In [21]:
# =====================================
# Save FAISS Index
# =====================================

faiss.write_index(
    index,
    "models/user_faiss.index"
)

print("FAISS Index Saved Successfully!")

FAISS Index Saved Successfully!


# Verifying the Saved Index

Finally, we verify that the FAISS index has been successfully saved.

This ensures that future notebooks can directly load the index without rebuilding it.

In [22]:
import os

print(os.listdir("models"))

['item_mapping.csv', 'item_embeddings.csv', 'user_faiss.index', 'user_mapping.csv', 'user_embeddings.csv']
